# T3 - Etapa 2: Análise exploratória e processamento de dados com Pandas
## SIASUS: produção ambulatorial dos estabelecimentos de Ijuí

**Fonte de dados (Estratégia A):** a VIEW `pa_ijui` (`sql/07_view_pa_ijui.sql`). Ela faz as ligações com as
tabelas descritivas e reduz a base aos estabelecimentos de Ijuí (`pa_ufmun = '431020'`).

**Período:** arquivos PARS de junho, julho e agosto de 2026. Na VIEW, `Ano` e `Mes` vêm da tabela
DIMTEMPO ligada pelo **mês de processamento** (`pa_mvm`), ou seja, o mês do arquivo em que o registro chegou.

Os resultados de cada análise ficam em DataFrames e também são gravados em `resultados/etapa2/`
para a Etapa 3 (visualização).

## 1. Conexão com o MySQL e carga da VIEW
Conexão com `mysql-connector-python` ao banco `dbsia`, no servidor local, com o usuário `root`. Se o
usuário tiver senha, ela deve ser informada em `SENHA`.

In [1]:
from pathlib import Path

import pandas as pd
from mysql.connector import connect

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

SENHA = ""
REMOVER_DUPLICADOS = False  # ver a seção 3.2 antes de mudar
SAIDA = Path(__file__).resolve().parents[1] / "resultados" / "etapa2" if "__file__" in globals() else Path("../resultados/etapa2")
SAIDA.mkdir(parents=True, exist_ok=True)


def consultar(sql):
    """Executa uma consulta no MySQL e devolve um DataFrame."""
    with connect(host="localhost", user="root", password=SENHA, database="dbsia") as connection:
        with connection.cursor() as cursor:
            cursor.execute(sql)
            return pd.DataFrame(cursor.fetchall(), columns=cursor.column_names)


def salvar(df, nome):
    """Grava o resultado em CSV (abre direto no Excel) e devolve o próprio DataFrame."""
    df.to_csv(SAIDA / f"{nome}.csv", encoding="utf-8-sig")
    return df


tabela = consultar("SELECT * FROM pa_ijui")
display(tabela.head(10)) if "display" in globals() else print(tabela.head(10))

    Ano  Mes AnoBimestre AnoTrimestre                                    Estabelecimento                                       Procedimento                                       Ocupacao pa_sexo  \
0  2026    6      2026B3       2026T2      CENTRO DE ATENCAO PSICOSSOCIAL CAPS DO BOSQUE  CONSULTA DE PROFISSIONAIS DE NIVEL SUPERIOR NA...                              Assistente social       0   
1  2026    6      2026B3       2026T2                                               CISA                 ULTRASSONOGRAFIA MAMARIA BILATERAL  Medico em radiologia e diagnostico por imagem       F   
2  2026    6      2026B3       2026T2                           HOSPITAL BOM PASTOR IJUI                                         TONOMETRIA                          Medico oftalmologista       M   
3  2026    6      2026B3       2026T2                           HOSPITAL BOM PASTOR IJUI                                         TONOMETRIA                          Medico oftalmologista       M   
4  2026   

## 2. Análise exploratória
### 2.1 Dimensão: quantos registros e variáveis existem?

In [2]:
print("Quantidade de linhas:", len(tabela))
print("Quantidade de colunas:", tabela.shape[1])
print("Colunas disponíveis:", list(tabela.columns))

Quantidade de linhas: 204301
Quantidade de colunas: 17
Colunas disponíveis: ['Ano', 'Mes', 'AnoBimestre', 'AnoTrimestre', 'Estabelecimento', 'Procedimento', 'Ocupacao', 'pa_sexo', 'pa_idade', 'pa_cidpri', 'Diagnostico', 'MunicipioEstabelecimento', 'MunicipioPaciente', 'pa_qtdpro', 'pa_qtdapr', 'pa_valpro', 'pa_valapr']


### 2.2 Tipos de variáveis
Quantidades e valores chegam do MySQL como `Decimal` e precisam ser convertidos para número.
A idade chega como texto (`'086'`).

In [3]:
colunas_numericas = ["pa_qtdpro", "pa_qtdapr", "pa_valpro", "pa_valapr"]
for coluna in colunas_numericas:
    tabela[coluna] = pd.to_numeric(tabela[coluna])
tabela["IdadeOriginal"] = tabela["pa_idade"]
tabela["IdadeNumerica"] = pd.to_numeric(tabela["pa_idade"], errors="coerce")

tipos = pd.DataFrame({"tipo_pandas": tabela.dtypes.astype(str), "valores_distintos": tabela.nunique()})
tipos["classificacao"] = "textual"
tipos.loc[tipos.valores_distintos <= 50, "classificacao"] = "categórica"
tipos.loc[colunas_numericas + ["IdadeNumerica", "Ano", "Mes"], "classificacao"] = "numérica"
salvar(tipos, "01_tipos_variaveis")
tipos

,tipo_pandas,valores_distintos,classificacao
Ano,int64,1,numérica
Mes,int64,3,numérica
AnoBimestre,str,2,categórica
AnoTrimestre,str,2,categórica
Estabelecimento,str,22,categórica
Procedimento,str,964,textual
Ocupacao,str,72,textual
pa_sexo,str,3,categórica
pa_idade,str,104,textual
pa_cidpri,str,785,textual


### 2.3 Estatísticas descritivas e distribuições de frequência

In [4]:
descritivas = tabela[colunas_numericas + ["IdadeNumerica"]].describe().T
descritivas["mediana"] = tabela[colunas_numericas + ["IdadeNumerica"]].median()
salvar(descritivas, "02_estatisticas_descritivas")
descritivas

,count,mean,std,min,25%,50%,75%,max,mediana
pa_qtdpro,"204,301.00",6.62,47.21,0.00,1.00,1.00,1.00,"6,284.00",1.00
pa_qtdapr,"204,301.00",5.23,43.07,0.00,0.00,1.00,1.00,"5,337.00",1.00
pa_valpro,"204,301.00",85.69,533.93,0.00,3.37,6.35,17.67,"60,741.37",6.35
pa_valapr,"204,301.00",82.90,533.29,0.00,0.00,3.37,13.51,"60,741.37",3.37
IdadeNumerica,"204,301.00",65.89,111.86,0.00,40.00,59.00,71.00,999.00,59.00


In [5]:
for coluna in ["Mes", "pa_sexo", "Estabelecimento", "MunicipioPaciente"]:
    print(f"\n{coluna} (10 mais frequentes):")
    print(tabela[coluna].value_counts(dropna=False).head(10).to_string())


Mes (10 mais frequentes):
Mes
8    75192
7    68827
6    60282

pa_sexo (10 mais frequentes):
pa_sexo
F    100686
M     92966
0     10649

Estabelecimento (10 mais frequentes):
Estabelecimento
HOSPITAL DE CLINICAS IJUI                                72608
HOSPITAL BOM PASTOR IJUI                                 57196
17 COORDENADORIA REGIONAL DE SAUDE                       17799
CER III UNIR                                             11115
APAE IJUI                                                 8612
CENTRO DE ATENCAO PSICOSSOCIAL CAPS DO BOSQUE             8330
CENTRO AUDITIVO PROAUDI                                   6417
CISA                                                      5548
CENTRO DE ATENCAO PSICOSSOCIAL CAPS INFANTIL CARAMELO     5488
POSTO CENTRAL SECRETARIA MUNICIPAL DE SAUDE DE IJUI       5394

MunicipioPaciente (10 mais frequentes):
MunicipioPaciente
IJUI               95541
INVALIDO           10649
PANAMBI             9251
JOIA                7436
CATUIPE          

### 2.4 Valores ausentes e inválidos
No SIASUS a falta de informação quase nunca aparece como `NULL`: ela vem codificada
(`0000` no CID, `999` na idade, `0` no sexo, município `INVALIDO`).

In [6]:
nulos = tabela.isna().sum()
problemas = pd.DataFrame({
    "registros": {
        "Diagnostico nulo (CID inexistente na tabela CID-10)": int(nulos["Diagnostico"]),
        "Procedimento nulo (código fora da tabela)": int(nulos["Procedimento"]),
        "pa_cidpri = '0000' (CID não informado)": int((tabela.pa_cidpri == "0000").sum()),
        "idade > 120 (idade inválida, geralmente 999)": int((tabela.IdadeNumerica > 120).sum()),
        "pa_sexo diferente de M/F": int((~tabela.pa_sexo.isin(["M", "F"])).sum()),
        "MunicipioPaciente = 'INVALIDO' (residência ignorada)": int((tabela.MunicipioPaciente == "INVALIDO").sum()),
        "valor aprovado maior que o apresentado": int((tabela.pa_valapr > tabela.pa_valpro).sum()),
    }
})
problemas["percentual"] = 100 * problemas.registros / len(tabela)
salvar(problemas, "03_ausentes_invalidos")
problemas

,registros,percentual
Diagnostico nulo (CID inexistente na tabela CID-10),24,0.01
Procedimento nulo (código fora da tabela),0,0.00
pa_cidpri = '0000' (CID não informado),137438,67.27
"idade > 120 (idade inválida, geralmente 999)",2774,1.36
pa_sexo diferente de M/F,10649,5.21
MunicipioPaciente = 'INVALIDO' (residência ignorada),10649,5.21
valor aprovado maior que o apresentado,354,0.17


Sexo, idade e residência ignorados aparecem juntos: são registros do **BPA consolidado**, que não trazem
dados do paciente. A VIEW não tem a coluna do tipo de registro (`pa_docorig`), mas o padrão aparece
cruzando as colunas:

In [7]:
pd.crosstab(tabela.pa_sexo, tabela.MunicipioPaciente == "INVALIDO", margins=True).rename(
    columns={True: "residência INVALIDO", False: "residência informada"})

MunicipioPaciente,residência informada,residência INVALIDO,All
pa_sexo,,,
0,0,10649,10649
F,100686,0,100686
M,92966,0,92966
All,193652,10649,204301


## 3. Limpeza e preparação
### 3.1 Estratégia para valores nulos
* **Nenhuma linha é apagada**: registros sem dados do paciente têm quantidade e valor válidos, e apagá-los
  subestimaria a produção.
* Códigos de "ignorado" viram **"Não informado"** nas colunas de texto e `NA` na idade.

In [8]:
df = tabela.copy()
df.loc[df.IdadeNumerica > 120, "IdadeNumerica"] = pd.NA
df["Sexo"] = df.pa_sexo.map({"M": "Masculino", "F": "Feminino"}).fillna("Não informado")
df["Diagnostico"] = df.Diagnostico.where(df.pa_cidpri != "0000", "Não informado").fillna("CID inexistente")
df["MunicipioPaciente"] = df.MunicipioPaciente.replace("INVALIDO", "Não informado")
df["Procedimento"] = df.Procedimento.fillna("Não informado")
df["FaixaEtaria"] = pd.cut(df.IdadeNumerica, bins=[-1, 14, 29, 44, 59, 200],
                           labels=["0–14", "15–29", "30–44", "45–59", "60+"])
df["Periodo"] = df.Ano.astype(str) + "-" + df.Mes.astype(str).str.zfill(2)

### 3.2 Registros duplicados
A VIEW tem só 17 das 62 colunas do PARS, e o PARS não identifica o paciente nem a data do atendimento.
Por isso há muitas linhas iguais que são atendimentos diferentes (por exemplo, várias sessões do mesmo
procedimento no mesmo mês). A remoção fica desligada (`REMOVER_DUPLICADOS = False`); a tabela mostra o
efeito que ela teria.

In [9]:
duplicadas = tabela.duplicated()
impacto_duplicados = pd.DataFrame({
    "com_duplicados": [len(df), df.pa_qtdpro.sum(), df.pa_valapr.sum()],
    "sem_duplicados": [(~duplicadas).sum(), df.loc[~duplicadas, "pa_qtdpro"].sum(),
                       df.loc[~duplicadas, "pa_valapr"].sum()],
}, index=["registros", "quantidade apresentada", "valor aprovado (R$)"])
impacto_duplicados["variacao_%"] = 100 * (impacto_duplicados.sem_duplicados / impacto_duplicados.com_duplicados - 1)
salvar(impacto_duplicados, "04_impacto_duplicados")
if REMOVER_DUPLICADOS:
    df = df[~duplicadas].copy()
impacto_duplicados

,com_duplicados,sem_duplicados,variacao_%
registros,"204,301.00","124,345.00",-39.14
quantidade apresentada,"1,351,467.00","1,223,997.00",-9.43
valor aprovado (R$),"16,936,148.85","15,759,192.92",-6.95


### 3.3 Classificação dos procedimentos por área
A VIEW traz o **nome** do procedimento, não o código. A área é identificada por palavras no nome
(`str.contains`), na ocupação e no estabelecimento. É uma aproximação: nomes que não seguem o padrão
ficam em "Outros".

In [10]:
AREAS = [  # a primeira regra que combinar define a área
    ("Oncologia", "Procedimento", r"QUIMIOTERAPIA|RADIOTERAPIA|HORMONIOTERAPIA|BRAQUITERAPIA|BETATERAPIA|ONCOLOG"),
    ("Saúde mental", "Ocupacao", r"(?i)psiquiatr|psicolog"),
    ("Saúde mental", "Estabelecimento", r"CAPS|PSICOSSOCIAL"),
    ("Medicamentos", "Procedimento", r"\(POR (COMPRIMIDO|CAPSULA|FRASCO|AMPOLA|SERINGA|ADESIVO|DRAGEA|SACHE|TUBO|BISNAGA|FRASCO-AMPOLA)|\d+ ?(MG|MCG|UI)\b"),
    ("Exames de imagem", "Procedimento", r"RADIOGRAFIA|ULTRASSONOGRAFIA|TOMOGRAFIA|RESSONANCIA|CINTILOGRAFIA|MAMOGRAFIA|ECOCARDIOGRAFIA|DENSITOMETRIA"),
    ("Exames laboratoriais", "Procedimento", r"^DOSAGEM|^HEMOGRAMA|^PESQUISA DE|^DETERMINACAO|^CONTAGEM|^ANALISE|^CULTURA|^EXAME|^ANTIBIOGRAMA|^TESTE"),
    ("Consultas e atendimentos", "Procedimento", r"^CONSULTA|^ATENDIMENTO|^ACOMPANHAMENTO|^ADMINISTRACAO"),
]
df["Area"] = "Outros"
for area, coluna, padrao in reversed(AREAS):  # de trás para frente: a primeira regra da lista prevalece
    df.loc[df[coluna].fillna("").str.contains(padrao, regex=True), "Area"] = area
df.Area.value_counts()

C:\Users\paracla\AppData\Local\Temp\ipykernel_9172\2306801933.py:12: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df.loc[df[coluna].fillna("").str.contains(padrao, regex=True), "Area"] = area


Area
Consultas e atendimentos    71446
Outros                      59811
Saúde mental                21746
Medicamentos                17701
Exames laboratoriais        14433
Exames de imagem            12844
Oncologia                    6320
Name: count, dtype: int64

### 3.4 Granularidade temporal
Mês, bimestre e trimestre vêm da DIMTEMPO (script `04-gera-dimtempo-mysql.sql`), ligada pelo mês de
processamento. **Atenção:** o arquivo de agosto também traz atendimentos de junho e julho processados com
atraso, que a VIEW conta como agosto.

In [11]:
tempo = df.groupby(["Ano", "Mes", "AnoBimestre", "AnoTrimestre"], as_index=False).size()
tempo

,Ano,Mes,AnoBimestre,AnoTrimestre,size
0,2026,6,2026B3,2026T2,60282
1,2026,7,2026B4,2026T3,68827
2,2026,8,2026B4,2026T3,75192


## 4. Análises estratégicas para o gestor
### 4.1 Volume e perfil dos procedimentos
Último mês, bimestre e trimestre disponíveis. Os bimestres e trimestres são os do calendário (DIMTEMPO);
como só há dados de junho a agosto, o 2º trimestre tem só junho e o 3º só julho e agosto.

In [12]:
ultimo_mes = df.Periodo.max()
ultimo_bi = df.AnoBimestre.max()
ultimo_tri = df.AnoTrimestre.max()
volume_periodos = pd.DataFrame({
    nome: {
        "registros": int(filtro.sum()),
        "quantidade apresentada": df.loc[filtro, "pa_qtdpro"].sum(),
        "quantidade aprovada": df.loc[filtro, "pa_qtdapr"].sum(),
        "meses com dados": ", ".join(sorted(df.loc[filtro, "Periodo"].unique())),
    }
    for nome, filtro in [(f"último mês ({ultimo_mes})", df.Periodo == ultimo_mes),
                         (f"último bimestre ({ultimo_bi})", df.AnoBimestre == ultimo_bi),
                         (f"último trimestre ({ultimo_tri})", df.AnoTrimestre == ultimo_tri)]
}).T
salvar(volume_periodos, "05_volume_periodos")
volume_periodos

,registros,quantidade apresentada,quantidade aprovada,meses com dados
último mês (2026-08),75192,"437,816.00","339,798.00",2026-08
último bimestre (2026B4),144019,"898,875.00","704,477.00","2026-07, 2026-08"
último trimestre (2026T3),144019,"898,875.00","704,477.00","2026-07, 2026-08"


In [13]:
por_area = df.groupby("Area")[["pa_qtdpro", "pa_qtdapr", "pa_valapr"]].sum().sort_values("pa_qtdpro", ascending=False)
por_area["participacao_%"] = 100 * por_area.pa_qtdpro / por_area.pa_qtdpro.sum()
salvar(por_area, "06_volume_por_area")
por_area

,pa_qtdpro,pa_qtdapr,pa_valapr,participacao_%
Area,,,,
Medicamentos,"775,804.00","586,464.00","2,614,432.76",57.40
Consultas e atendimentos,"183,957.00","144,935.00","1,132,938.49",13.61
Exames laboratoriais,"183,117.00","161,246.00","776,966.79",13.55
Outros,"135,741.00","112,622.00","5,703,750.64",10.04
Saúde mental,"36,877.00","30,206.00","127,842.09",2.73
Exames de imagem,"30,076.00","27,182.00","1,489,251.51",2.23
Oncologia,"5,895.00","5,844.00","5,090,966.57",0.44


In [14]:
top_procedimentos = (df.groupby("Procedimento")[["pa_qtdpro", "pa_qtdapr"]].sum()
                       .sort_values("pa_qtdpro", ascending=False).head(15))
salvar(top_procedimentos, "07_top_procedimentos")
top_procedimentos

,pa_qtdpro,pa_qtdapr
Procedimento,,
FORMOTEROL 12 MCG + BUDESONIDA 400 MCG Ç¨(POR CAPSULA INALANTE),"180,232.00","45,000.00"
GABAPENTINA 300 MG (POR CAPSULA),"82,420.00","45,000.00"
CONSULTA DE PROFISSIONAIS DE NIVEL SUPERIOR NA ATENCAO ESPECIALIZADA (EXCETO MEDICO),"69,602.00","53,963.00"
CONSULTA MEDICA EM ATENCAO ESPECIALIZADA,"56,395.00","46,672.00"
AZATIOPRINA 50 MG (POR COMPRIMIDO),"29,780.00","29,780.00"
CLOZAPINA 100 MG (POR COMPRIMIDO),"23,588.00","23,588.00"
ADMINISTRACAO DE MEDICAMENTOS NA ATENCAO ESPECIALIZADA.,"23,516.00","13,138.00"
ATORVASTATINA 40 MG (POR COMPRIMIDO),"21,756.00","21,756.00"
TACROLIMO 1 MG (POR CAPSULA),"20,290.00","20,290.00"


**Evolução temporal:** produção mensal por área e variação em relação ao mês anterior (`pct_change`).

In [15]:
evolucao_area = df.pivot_table(index="Area", columns="Periodo", values="pa_qtdpro", aggfunc="sum", fill_value=0)
variacao = evolucao_area.pct_change(axis=1).iloc[:, 1:].mul(100).add_prefix("var_%_")
evolucao_area = evolucao_area.join(variacao)
salvar(evolucao_area, "08_evolucao_por_area")
evolucao_area

Periodo,2026-06,2026-07,2026-08,var_%_2026-07,var_%_2026-08
Area,,,,,
Consultas e atendimentos,"64,541.00","43,476.00","75,940.00",-32.64,74.67
Exames de imagem,"10,176.00","7,372.00","12,528.00",-27.56,69.94
Exames laboratoriais,"71,092.00","74,546.00","37,479.00",4.86,-49.72
Medicamentos,"245,297.00","270,614.00","259,893.00",10.32,-3.96
Oncologia,"1,975.00","2,021.00","1,899.00",2.33,-6.04
Outros,"43,884.00","44,836.00","47,021.00",2.17,4.87
Saúde mental,"15,627.00","18,194.00","3,056.00",16.43,-83.20


### 4.2 Produção por estabelecimento de saúde
Ranking pela quantidade apresentada, com a taxa de aprovação. A 17ª Coordenadoria Regional de Saúde
dispensa medicamentos para toda a região, contados por comprimido; por isso também há um ranking sem ela.

In [16]:
ranking = (df.groupby("Estabelecimento")
             .agg(Registros=("Procedimento", "size"),
                  QuantidadeApresentada=("pa_qtdpro", "sum"), QuantidadeAprovada=("pa_qtdapr", "sum"),
                  ValorApresentado=("pa_valpro", "sum"), ValorAprovado=("pa_valapr", "sum"))
             .sort_values("QuantidadeApresentada", ascending=False))
ranking["Aprovacao_qtd_%"] = 100 * ranking.QuantidadeAprovada / ranking.QuantidadeApresentada
ranking["Glosa_qtd"] = ranking.QuantidadeApresentada - ranking.QuantidadeAprovada
salvar(ranking, "09_ranking_estabelecimentos")
ranking

,Registros,QuantidadeApresentada,QuantidadeAprovada,ValorApresentado,ValorAprovado,Aprovacao_qtd_%,Glosa_qtd
Estabelecimento,,,,,,,
17 COORDENADORIA REGIONAL DE SAUDE,17799,"784,902.00","595,562.00","2,632,581.82","2,622,471.70",75.88,"189,340.00"
HOSPITAL DE CLINICAS IJUI,72608,"138,151.00","114,505.00","10,034,386.01","9,692,270.79",82.88,"23,646.00"
CISA,5548,"94,333.00","75,999.00","683,148.16","514,376.90",80.56,"18,334.00"
POSTO CENTRAL SECRETARIA MUNICIPAL DE SAUDE DE IJUI,5394,"82,297.00","74,294.00","723,415.75","651,993.44",90.28,"8,003.00"
LABORATORIO DE ANALISES CLINICAS UNILAB,609,"57,272.00","57,272.00","282,563.77","282,563.77",100.00,0.00
HOSPITAL BOM PASTOR IJUI,57196,"57,214.00","28,115.00","870,428.70","924,967.03",49.14,"29,099.00"
UNIDADE DE PRONTO ATENDIMENTO DR MARTIN CESAR AGNOLETTO,501,"47,873.00","41,483.00","129,340.74","124,773.78",86.65,"6,390.00"
HEMOVITA,354,"16,199.00","16,199.00","82,765.63","82,765.63",100.00,0.00
CENTRO DE ATENCAO PSICOSSOCIAL CAPS DO BOSQUE,8330,"14,415.00","9,640.00","9,755.50","9,755.50",66.87,"4,775.00"


In [17]:
ranking_sem_medicamentos = (df[df.Area != "Medicamentos"].groupby("Estabelecimento")["pa_qtdpro"].sum()
                              .sort_values(ascending=False).head(10).to_frame())
salvar(ranking_sem_medicamentos, "10_ranking_sem_medicamentos")
ranking_sem_medicamentos

,pa_qtdpro
Estabelecimento,
HOSPITAL DE CLINICAS IJUI,"138,151.00"
CISA,"94,333.00"
POSTO CENTRAL SECRETARIA MUNICIPAL DE SAUDE DE IJUI,"82,297.00"
LABORATORIO DE ANALISES CLINICAS UNILAB,"57,272.00"
HOSPITAL BOM PASTOR IJUI,"57,214.00"
UNIDADE DE PRONTO ATENDIMENTO DR MARTIN CESAR AGNOLETTO,"47,873.00"
HEMOVITA,"16,199.00"
CENTRO DE ATENCAO PSICOSSOCIAL CAPS DO BOSQUE,"14,415.00"
CER III UNIR,"11,836.00"


### 4.3 Perfil demográfico e epidemiológico

In [18]:
sexo_faixa = df.pivot_table(index="FaixaEtaria", columns="Sexo", values="pa_qtdpro",
                            aggfunc="sum", fill_value=0, observed=False)
sexo_faixa.loc["Não informado"] = df[df.FaixaEtaria.isna()].groupby("Sexo")["pa_qtdpro"].sum()
sexo_faixa = sexo_faixa.fillna(0)
salvar(sexo_faixa, "11_sexo_faixa_etaria")
sexo_faixa

Sexo,Feminino,Masculino,Não informado
FaixaEtaria,,,
0–14,"15,420.00","22,347.00","29,087.00"
15–29,"36,639.00","29,885.00","16,200.00"
30–44,"63,567.00","65,654.00","16,474.00"
45–59,"149,403.00","96,690.00","15,304.00"
60+,"280,522.00","218,509.00","18,647.00"
Não informado,0.00,0.00,"277,119.00"


In [19]:
com_cid = df[~df.Diagnostico.isin(["Não informado", "CID inexistente"])]
print(f"Registros com CID informado: {len(com_cid):,} de {len(df):,} ({100 * len(com_cid) / len(df):.1f}%)")
top_cid = com_cid.groupby("Diagnostico")["pa_qtdpro"].sum().sort_values(ascending=False).head(15).to_frame()
salvar(top_cid, "12_top_diagnosticos")
top_cid

Registros com CID informado: 66,839 de 204,301 (32.7%)


,pa_qtdpro
Diagnostico,
J45.0 Asma predom alergica,"122,493.00"
R52.2 Outr dor cronica,"73,357.00"
G20 Doenc de Parkinson,"42,542.00"
J44.0 Doen pulm obs cron c/inf resp ag tr resp inf,"34,492.00"
J44.8 Outr form espec doenc pulmonar obstrut cron,"31,492.00"
F20.0 Esquizofrenia paranoide,"31,421.00"
M06.0 Artrite reumatoide soro-negativa,"28,811.00"
Z94.0 Rim transplantado,"26,769.00"
R52.1 Dor cronica intratavel,"21,394.00"


In [20]:
GRUPOS_CID = {
    "Hipertensão (I10-I15)": r"^I1[0-5]", "Diabetes (E10-E14)": r"^E1[0-4]",
    "Neoplasias (C00-D48)": r"^(C|D[0-3]|D4[0-8])", "Transtornos mentais (F00-F99)": r"^F",
    "Doenças do olho (H00-H59)": r"^H[0-5]", "Doenças do ouvido (H60-H95)": r"^H[6-9]",
    "Doença renal crônica (N18)": r"^N18",
}
grupos_cid = pd.DataFrame({
    nome: {"registros": int(com_cid.pa_cidpri.str.contains(rx).sum()),
           "quantidade apresentada": com_cid.loc[com_cid.pa_cidpri.str.contains(rx), "pa_qtdpro"].sum()}
    for nome, rx in GRUPOS_CID.items()
}).T.sort_values("quantidade apresentada", ascending=False)
salvar(grupos_cid, "13_grupos_diagnosticos")
grupos_cid

C:\Users\paracla\AppData\Local\Temp\ipykernel_9172\4156013307.py:8: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  nome: {"registros": int(com_cid.pa_cidpri.str.contains(rx).sum()),
C:\Users\paracla\AppData\Local\Temp\ipykernel_9172\4156013307.py:9: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  "quantidade apresentada": com_cid.loc[com_cid.pa_cidpri.str.contains(rx), "pa_qtdpro"].sum()}


,registros,quantidade apresentada
Transtornos mentais (F00-F99),"13,306.00","98,412.00"
Diabetes (E10-E14),"1,688.00","16,367.00"
Neoplasias (C00-D48),"7,994.00","8,716.00"
Doenças do ouvido (H60-H95),"5,962.00","6,804.00"
Doenças do olho (H00-H59),"3,040.00","4,272.00"
Doença renal crônica (N18),505.00,"4,122.00"
Hipertensão (I10-I15),7.00,7.00


### 4.4 Fluxos regionais e acesso
Residente é quem tem `MunicipioPaciente` igual ao município do estabelecimento (Ijuí). A VIEW traz só o
nome do município, sem o estado; alguns nomes existem em mais de um estado.

In [21]:
df["Origem"] = "Outro município"
df.loc[df.MunicipioPaciente == df.MunicipioEstabelecimento, "Origem"] = "Ijuí"
df.loc[df.MunicipioPaciente == "Não informado", "Origem"] = "Não informado"
origem = df.groupby("Origem").agg(Registros=("Procedimento", "size"), QuantidadeApresentada=("pa_qtdpro", "sum"))
origem["participacao_qtd_%"] = 100 * origem.QuantidadeApresentada / origem.QuantidadeApresentada.sum()
salvar(origem, "14_residentes_x_outros")
origem

,Registros,QuantidadeApresentada,participacao_qtd_%
Origem,,,
Ijuí,95541,"401,177.00",29.68
Não informado,10649,"372,831.00",27.59
Outro município,98111,"577,459.00",42.73


In [22]:
municipios_origem = (df[df.Origem == "Outro município"].groupby("MunicipioPaciente")["pa_qtdpro"].sum()
                       .sort_values(ascending=False).to_frame())
municipios_origem["participacao_%"] = 100 * municipios_origem.pa_qtdpro / municipios_origem.pa_qtdpro.sum()
print("Municípios de origem diferentes:", len(municipios_origem))
salvar(municipios_origem, "15_municipios_origem")
municipios_origem.head(15)

Municípios de origem diferentes: 244


,pa_qtdpro,participacao_%
MunicipioPaciente,,
PANAMBI,"138,632.00",24.01
CRISSIUMAL,"51,968.00",9.00
JOIA,"39,812.00",6.89
CONDOR,"35,993.00",6.23
CATUIPE,"35,223.00",6.10
SANTO AUGUSTO,"34,274.00",5.94
AUGUSTO PESTANA,"31,050.00",5.38
AJURICABA,"27,452.00",4.75
CORONEL BARROS,"20,543.00",3.56


In [23]:
origem_por_area = df.pivot_table(index="Area", columns="Origem", values="pa_qtdpro", aggfunc="sum", fill_value=0)
origem_por_area["outros_municipios_%"] = 100 * origem_por_area["Outro município"] / origem_por_area.sum(axis=1)
origem_por_area = origem_por_area.sort_values("outros_municipios_%", ascending=False)
salvar(origem_por_area, "16_origem_por_area")
origem_por_area

Origem,Ijuí,Não informado,Outro município,outros_municipios_%
Area,,,,
Oncologia,"1,190.00",0.00,"4,705.00",79.81
Medicamentos,"300,369.00",0.00,"475,435.00",61.28
Outros,"35,215.00","57,387.00","43,139.00",31.78
Exames de imagem,"4,437.00","17,102.00","8,537.00",28.38
Consultas e atendimentos,"29,187.00","117,678.00","37,092.00",20.16
Exames laboratoriais,"6,952.00","169,045.00","7,120.00",3.89
Saúde mental,"23,827.00","11,619.00","1,431.00",3.88


### 4.5 Recursos financeiros

In [24]:
financeiro = df.groupby("Periodo")[["pa_valpro", "pa_valapr", "pa_qtdpro", "pa_qtdapr"]].sum()
financeiro.loc["Total"] = financeiro.sum()
financeiro["Diferenca_valor"] = financeiro.pa_valpro - financeiro.pa_valapr
financeiro["Aprovado_%"] = 100 * financeiro.pa_valapr / financeiro.pa_valpro
financeiro["Gasto_medio_por_procedimento"] = financeiro.pa_valapr / financeiro.pa_qtdapr
salvar(financeiro, "17_financeiro_mensal")
financeiro

,pa_valpro,pa_valapr,pa_qtdpro,pa_qtdapr,Diferenca_valor,Aprovado_%,Gasto_medio_por_procedimento
Periodo,,,,,,,
2026-06,"5,557,640.99","5,436,537.16","452,592.00","364,022.00","121,103.83",97.82,14.93
2026-07,"6,272,060.94","6,158,024.69","461,059.00","364,679.00","114,036.25",98.18,16.89
2026-08,"5,676,178.26","5,341,587.00","437,816.00","339,798.00","334,591.26",94.11,15.72
Total,"17,505,880.19","16,936,148.85","1,351,467.00","1,068,499.00","569,731.34",96.75,15.85


In [25]:
gasto_area = df.pivot_table(index="Area", columns="Periodo", values=["pa_valapr", "pa_qtdapr"], aggfunc="sum")
gasto_medio_area = (gasto_area["pa_valapr"] / gasto_area["pa_qtdapr"]).round(2)
salvar(gasto_medio_area, "18_gasto_medio_por_area")
gasto_medio_area

Periodo,2026-06,2026-07,2026-08
Area,,,
Consultas e atendimentos,7.62,8.18,7.78
Exames de imagem,54.54,71.86,45.23
Exames laboratoriais,4.79,4.87,4.78
Medicamentos,4.25,4.50,4.61
Oncologia,845.26,942.15,821.95
Outros,44.93,59.71,47.49
Saúde mental,3.61,3.46,10.94


### 4.6 Áreas críticas
#### Oncologia: quimioterapia x radioterapia (apresentado x aprovado)

In [26]:
onco = df[df.Area == "Oncologia"].copy()
onco["Modalidade"] = "Quimioterapia/hormonioterapia"
onco.loc[onco.Procedimento.str.contains("RADIOTERAPIA|BRAQUITERAPIA|BETATERAPIA"), "Modalidade"] = "Radioterapia"
oncologia = onco.groupby(["Modalidade", "Periodo"])[["pa_qtdpro", "pa_qtdapr", "pa_valpro", "pa_valapr"]].sum()
oncologia["Aprovacao_qtd_%"] = 100 * oncologia.pa_qtdapr / oncologia.pa_qtdpro
salvar(oncologia, "19_oncologia")
print("Estabelecimentos com produção oncológica:", onco.Estabelecimento.unique().tolist())
oncologia

Estabelecimentos com produção oncológica: ['HOSPITAL DE CLINICAS IJUI']


pa_qtdpro  pa_qtdapr    pa_valpro    pa_valapr  Aprovacao_qtd_%
Modalidade                    Periodo                                                                 
Quimioterapia/hormonioterapia 2026-06   1,788.00   1,740.00 1,027,785.99 1,027,561.83            97.32
                              2026-07   1,785.00   1,783.00 1,082,220.45 1,082,136.03            99.89
                              2026-08   1,735.00   1,735.00 1,038,230.21 1,038,230.21           100.00
Radioterapia                  2026-06     187.00     187.00   601,254.30   601,254.30           100.00
                              2026-07     236.00     235.00   822,873.30   819,131.70            99.58
                              2026-08     164.00     164.00   522,652.50   522,652.50           100.00

#### Saúde mental: atendimentos psiquiátricos e psicológicos

In [27]:
sm = df[df.Area == "Saúde mental"].copy()
sm["Profissional"] = "Outros profissionais (CAPS)"
sm.loc[sm.Ocupacao.str.contains("(?i)psiquiatr"), "Profissional"] = "Psiquiatra"
sm.loc[sm.Ocupacao.str.contains("(?i)psicolog"), "Profissional"] = "Psicólogo"
saude_mental = sm.pivot_table(index=["Estabelecimento", "Profissional"], columns="Periodo",
                              values="pa_qtdpro", aggfunc="sum", fill_value=0)
saude_mental["Total"] = saude_mental.sum(axis=1)
saude_mental = saude_mental.sort_values("Total", ascending=False)
salvar(saude_mental, "20_saude_mental")
saude_mental

,Periodo,2026-06,2026-07,2026-08,Total
Estabelecimento,Profissional,,,,
CENTRO DE ATENCAO PSICOSSOCIAL CAPS DO BOSQUE,Outros profissionais (CAPS),"5,496.00","6,864.00",0.00,"12,360.00"
CENTRO DE ATENCAO PSICOSSOCIAL CAPS INFANTIL CARAMELO,Outros profissionais (CAPS),"2,161.00","2,517.00",0.00,"4,678.00"
CENTRO DE ATENCAO PSICOSSOCIAL CAPS COLMEIA,Outros profissionais (CAPS),"2,028.00","2,452.00",0.00,"4,480.00"
APAE IJUI,Psicólogo,"1,173.00","1,242.00","1,072.00","3,487.00"
CENTRO DE ATENCAO PSICOSSOCIAL CAPS INFANTIL CARAMELO,Psicólogo,"1,114.00","1,346.00",0.00,"2,460.00"
CENTRO DE ATENCAO PSICOSSOCIAL CAPS DO BOSQUE,Psicólogo,835.00,"1,141.00",0.00,"1,976.00"
CISA,Psiquiatra,487.00,510.00,414.00,"1,411.00"
CER III UNIR,Psicólogo,438.00,422.00,430.00,"1,290.00"
CENTRO DE ATENCAO PSICOSSOCIAL CAPS COLMEIA,Psiquiatra,550.00,519.00,0.00,"1,069.00"


#### Atenção básica
**Limitação:** as UBS registram sua produção no e-SUS APS / SISAB, não no SIASUS, e a VIEW não traz o tipo
de estabelecimento. Ficam a produção do Posto Central (unidade municipal) e o acompanhamento de doenças
crônicas (hipertensão e diabetes) em qualquer estabelecimento.

In [28]:
posto = df[df.Estabelecimento.str.contains("POSTO|UBS|UNIDADE BASICA|ESF", na=False)]
atencao_basica = posto.pivot_table(index="Area", columns="Periodo", values="pa_qtdpro", aggfunc="sum", fill_value=0)
salvar(atencao_basica, "21_atencao_basica")
print("Estabelecimentos considerados:", posto.Estabelecimento.unique().tolist())
atencao_basica

Estabelecimentos considerados: ['POSTO CENTRAL SECRETARIA MUNICIPAL DE SAUDE DE IJUI']


Periodo,2026-06,2026-08
Area,,
Consultas e atendimentos,"23,390.00","38,547.00"
Exames de imagem,"2,678.00","5,724.00"
Exames laboratoriais,132.00,230.00
Outros,"3,856.00","6,832.00"
Saúde mental,131.00,777.00


In [29]:
cronicas = com_cid[com_cid.pa_cidpri.str.contains(r"^(I1[0-5]|E1[0-4])")].copy()
cronicas["Condicao"] = cronicas.pa_cidpri.str.startswith("I").map({True: "Hipertensão", False: "Diabetes"})
doencas_cronicas = cronicas.pivot_table(index=["Condicao", "Estabelecimento"], columns="Periodo",
                                        values="pa_qtdpro", aggfunc="sum", fill_value=0)
salvar(doencas_cronicas, "22_doencas_cronicas")
doencas_cronicas

C:\Users\paracla\AppData\Local\Temp\ipykernel_9172\925495395.py:1: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  cronicas = com_cid[com_cid.pa_cidpri.str.contains(r"^(I1[0-5]|E1[0-4])")].copy()


Periodo                                         2026-06  2026-07  2026-08
Condicao    Estabelecimento                                              
Diabetes    17 COORDENADORIA REGIONAL DE SAUDE 5,280.00 5,492.00 5,490.00
            CER III UNIR                          44.00     7.00    54.00
Hipertensão HOSPITAL BOM PASTOR IJUI               0.00     1.00     0.00
            HOSPITAL DE CLINICAS IJUI              3.00     0.00     3.00

## 5. Base preparada para a Etapa 3
O DataFrame tratado é gravado em formato Parquet para a etapa de visualização.

In [30]:
df.to_parquet(SAIDA / "pa_ijui_preparado.parquet", index=False)
print(f"Base preparada: {len(df):,} registros, {df.shape[1]} colunas -> {SAIDA / 'pa_ijui_preparado.parquet'}")

Base preparada: 204,301 registros, 24 colunas -> ..\resultados\etapa2\pa_ijui_preparado.parquet
